# Laboratorio 08: consolidación de APIs biomédicas

Este notebook integra los registros validados de
ClinicalTrials.gov, PubMed y openFDA para la condición
**cáncer gástrico**.

Los resultados corresponden a una muestra pequeña de tres
registros por fuente. Por ello, las conclusiones demuestran
el funcionamiento técnico del pipeline y no representan
estimaciones epidemiológicas ni clínicas generales.

## 1. Cargar los archivos Parquet

Los archivos se generaron únicamente después de conservar
las respuestas crudas, adaptar sus estructuras y validar
cada registro mediante Pydantic.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    candidates = [start, *start.parents]

    for candidate in candidates:
        if (
            (candidate / "pyproject.toml").exists()
            and (candidate / "clientes").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "No se encontró la raíz del Laboratorio 08."
    )


project_root = find_project_root(
    Path.cwd().resolve()
)
processed = project_root / "data" / "processed"

trials = pd.read_parquet(
    processed / "clinicaltrials.parquet"
)
articles = pd.read_parquet(
    processed / "pubmed.parquet"
)
events = pd.read_parquet(
    processed / "openfda.parquet"
)

print("ClinicalTrials.gov:", trials.shape)
print("PubMed:", articles.shape)
print("openFDA:", events.shape)

ClinicalTrials.gov: (3, 8)
PubMed: (3, 5)
openFDA: (3, 7)


## 2. Inspección de los datos normalizados

In [2]:
display(
    trials[
        [
            "nct_id",
            "overall_status",
            "enrollment_count",
            "countries",
        ]
    ]
)

display(
    articles[
        [
            "pmid",
            "publication_year",
            "journal",
            "title",
        ]
    ]
)

display(
    events[
        [
            "safety_report_id",
            "serious",
            "drugs",
            "reactions",
        ]
    ]
)

,nct_id,overall_status,enrollment_count,countries
0,NCT00911820,COMPLETED,88,[United States]
1,NCT03531931,UNKNOWN,20,[China]
2,NCT01783951,COMPLETED,63,[China]


,pmid,publication_year,journal,title
0,42829718,2026,OncoTargets and therapy,Spatial Fibroblast-Macrophage Niches in Gastri...
1,42828570,2026,Gastric cancer : official journal of the Inter...,From operating room to pathology report: Delph...
2,42828177,2026,Frontiers in immunology,Deep learning-based radiomics and pathomics fo...


,safety_report_id,serious,drugs,reactions
0,10004141,True,[GLEEVEC],"[Hepatic lesion, Malignant neoplasm progressio..."
1,10004413,True,"[DOCETAXEL (UNKNOWN), ENALAPRIL MALEATE]","[Facial bones fracture, Syncope, Neutropenia, ..."
2,10005256,True,"[LANSOPRAZOLE, PERINDOPRIL, EPIRUBICIN, AVASTI...","[Chest pain, Pyrexia]"


## 3. Pregunta 1: ¿cuántos ensayos activos hay y dónde?

Se consideran activos los estados `RECRUITING`,
`NOT_YET_RECRUITING`, `ENROLLING_BY_INVITATION` y
`ACTIVE_NOT_RECRUITING`.

In [3]:
active_statuses = {
    "ACTIVE_NOT_RECRUITING",
    "ENROLLING_BY_INVITATION",
    "NOT_YET_RECRUITING",
    "RECRUITING",
}

active_trials = trials.loc[
    trials["overall_status"].isin(active_statuses),
    [
        "nct_id",
        "overall_status",
        "countries",
    ],
].explode(
    "countries",
    ignore_index=True,
)

active_by_country = (
    active_trials["countries"]
    .dropna()
    .value_counts()
    .rename_axis("country")
    .reset_index(name="active_trials")
)

if active_by_country.empty:
    print(
        "No hay ensayos activos en la muestra validada."
    )
else:
    display(active_by_country)

No hay ensayos activos en la muestra validada.


**Resultado:** ninguno de los tres ensayos de la muestra
presentó un estado considerado activo. Dos estaban
completados y uno tenía estado desconocido. Este resultado
solo describe la muestra utilizada.

## 4. Pregunta 2: ¿cuáles fueron las reacciones más reportadas?

Las listas de reacciones se expanden para contar cada
término de manera independiente.

In [4]:
reaction_counts = (
    events[
        [
            "safety_report_id",
            "reactions",
        ]
    ]
    .explode(
        "reactions",
        ignore_index=True,
    )["reactions"]
    .dropna()
    .value_counts()
    .rename_axis("reaction")
    .reset_index(name="reports")
)

display(reaction_counts)

,reaction,reports
0,Pyrexia,2
1,Hepatic lesion,1
2,Malignant neoplasm progression,1
3,Gastric cancer,1
4,Product use issue,1
5,Facial bones fracture,1
6,Syncope,1
7,Neutropenia,1
8,Chest pain,1


**Resultado:** `Pyrexia` apareció en dos reportes. Las
demás reacciones aparecieron una vez. Los reportes de
openFDA son notificaciones de farmacovigilancia y no
demuestran causalidad entre un medicamento y una reacción.

## 5. Pregunta 3: ¿cómo se distribuyen las publicaciones por año?

In [5]:
publications_by_year = (
    articles["publication_year"]
    .value_counts()
    .sort_index()
    .rename_axis("publication_year")
    .reset_index(name="articles")
)

display(publications_by_year)

,publication_year,articles
0,2026,3


**Resultado:** los tres artículos recuperados fueron
publicados en 2026. La muestra se obtuvo de los primeros
resultados de una consulta ordenada por la API y no debe
interpretarse como la evolución histórica completa de la
literatura sobre cáncer gástrico.

## 6. Conclusiones

El pipeline integró correctamente tres fuentes con
estructuras distintas y produjo tablas Parquet que pueden
reutilizarse sin consultar nuevamente las APIs.

En esta muestra no hubo ensayos activos, `Pyrexia` fue la
única reacción repetida y los tres artículos correspondieron
a 2026. La utilidad principal de este análisis es demostrar
una ruta reproducible que posteriormente puede ampliarse a
consultas por genes, pronóstico y regulación epigenética.